In [21]:
import pandas as pd
import numpy as np

file_path = "C:/Users/RSCH/Desktop/DWD/Final project/Retail-Intelligence-Platform/datasets/amez_dataset_cleaned.xlsx"

data = pd.read_excel(
    file_path,
    sheet_name="Overall_Transactions"
)

data.head()

,InvoiceID,InvoiceDate,CustomerID,ProductID,ProductName,Category,SubCategory,Quantity,UnitPrice,Discount,TotalAmount,StockQuantity,StoreID,StoreName
0,INV-2026-1001,2026-01-01,64d75846-3182-4000-057e-c1df000064d,8e4669ab-db13-4000-0bd7-f03200008e46,Merino Wool Minimalist Crewneck Knitwear,Fashion & Apparel,Men's Fashion,1,11500.0,2000,9500.0,75,9abc0803-cfe9-4000-0a98-919a00009abc,Luxe Apparel Studio
1,INV-2026-1001,2026-01-01,64d75846-3182-4000-057e-c1df000064d,93c93477-c69c-4000-0a0f-adee000093c9,Bio-Peptide Restorative Night Crème,Beauty & Wellness,Skincare Essentials,2,6800.0,2000,11600.0,115,9fbc0fe2-cae9-4000-0ac8-967b00009fbc,Glow Naturals Cosmetics
2,INV-2026-1001,2026-01-01,64d75846-3182-4000-057e-c1df000064d,7ec257a3-2b97-4000-04df-ce3a00007ec2,Velocity Pro Carbon Running Sneakers,Fashion & Apparel,Footwear,2,17500.0,4000,31000.0,95,a0bc1175-f5e9-4000-0938-88ec0000a0bc,Velocity Athletic Gear
3,INV-2026-1001,2026-01-01,64d75846-3182-4000-057e-c1df000064d,81c25c5c-d497-4000-0b2f-c5c5000081c2,Ergonomic Breathable Mesh Desk Chair,Home & Living,Modern Furniture,1,28000.0,4000,24000.0,50,99bc0670-cce9-4000-0aa8-9fe9000099bc,Nordic Living Essentials
4,INV-2026-1001,2026-01-01,64d75846-3182-4000-057e-c1df000064d,93c93477-c69c-4000-0a0f-adee000093c9,Bio-Peptide Restorative Night Crème,Beauty & Wellness,Skincare Essentials,1,6800.0,1000,5800.0,115,9fbc0fe2-cae9-4000-0ac8-967b00009fbc,Glow Naturals Cosmetics


In [22]:
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])

data["Year"] = data["InvoiceDate"].dt.year
data["Month"] = data["InvoiceDate"].dt.month
data["Day"] = data["InvoiceDate"].dt.day
data["DayOfWeek"] = data["InvoiceDate"].dt.dayofweek
data["WeekOfYear"] = data["InvoiceDate"].dt.isocalendar().week.astype(int)

In [23]:
data["Revenue"] = data["TotalAmount"]

In [24]:
data["EffectivePrice"] = (
    data["UnitPrice"] - 
    (data["Discount"] / data["Quantity"])
)

In [25]:
customer_features = data.groupby("CustomerID").agg(
    TotalOrders=("InvoiceID", "nunique"),
    TotalQuantity=("Quantity", "sum"),
    TotalRevenue=("TotalAmount", "sum"),
    UniqueProducts=("ProductID", "nunique"),
    AverageOrderValue=("TotalAmount", "mean")
).reset_index()

customer_features.head()

,CustomerID,TotalOrders,TotalQuantity,TotalRevenue,UniqueProducts,AverageOrderValue
0,0025d7a1-81d7-4240-8807-4aed48f88cde,1,8,258142.0,5,51628.400000
1,00381394-0d27-43e1-aa0c-2105ea482b2b,1,20,396249.0,6,66041.500000
2,003c8473-1b85-445f-bcfb-948e6c9f8451,1,16,375593.0,6,62598.833333
3,004d5aa3-b784-4619-8e18-0411b8f90404,1,13,162866.0,6,27144.333333
4,0089210b-880d-4835-b385-14486458934b,1,15,278088.0,6,46348.000000


In [26]:
customer_product = data.groupby(
    ["CustomerID", "ProductID"]
).agg(
    PurchaseCount=("InvoiceID", "nunique"),
    TotalQuantity=("Quantity", "sum"),
    TotalSpent=("TotalAmount", "sum")
).reset_index()

In [27]:
reference_date = data["InvoiceDate"].max() + pd.Timedelta(days=1)

recency = data.groupby("CustomerID")["InvoiceDate"].max().reset_index()

recency["Recency"] = (
    reference_date - recency["InvoiceDate"]
).dt.days

recency = recency[["CustomerID", "Recency"]]

In [28]:
customer_features = customer_features.merge(
    recency,
    on="CustomerID",
    how="left"
)

customer_features.head()

,CustomerID,TotalOrders,TotalQuantity,TotalRevenue,UniqueProducts,AverageOrderValue,Recency
0,0025d7a1-81d7-4240-8807-4aed48f88cde,1,8,258142.0,5,51628.400000,153
1,00381394-0d27-43e1-aa0c-2105ea482b2b,1,20,396249.0,6,66041.500000,184
2,003c8473-1b85-445f-bcfb-948e6c9f8451,1,16,375593.0,6,62598.833333,100
3,004d5aa3-b784-4619-8e18-0411b8f90404,1,13,162866.0,6,27144.333333,70
4,0089210b-880d-4835-b385-14486458934b,1,15,278088.0,6,46348.000000,88


In [29]:
product_features = data.groupby("ProductID").agg(
    TotalQuantity=("Quantity", "sum"),
    TotalRevenue=("TotalAmount", "sum"),
    AveragePrice=("EffectivePrice", "mean"),
    AverageDiscount=("Discount", "mean"),
    AverageStock=("StockQuantity", "mean")
).reset_index()

product_features.head()

,ProductID,TotalQuantity,TotalRevenue,AveragePrice,AverageDiscount,AverageStock
0,0128933d-9879-4346-aafc-a8045c36b087,208,6863643.0,32799.644195,1438.202247,120.629213
1,03df1a4a-b5a2-40ee-b621-f064b782d4a1,268,1521762.0,5476.479167,1552.083333,141.875000
2,04df7b21-ab56-460d-96c8-25aed6e602a0,221,1922586.0,8495.295880,1511.235955,88.719101
3,06a13e3b-0b08-40f7-bc7c-74fc8f197b7f,192,1425899.0,7281.497778,1666.666667,195.106667
4,0a7982e2-2a56-4dea-b738-821ef43ca9ae,184,2105824.0,11168.362869,1449.367089,137.481013


In [30]:
daily_demand = data.groupby(
    ["ProductID", data["InvoiceDate"].dt.date]
)["Quantity"].sum().reset_index()

daily_demand.columns = [
    "ProductID",
    "Date",
    "Demand"
]

daily_demand["Date"] = pd.to_datetime(daily_demand["Date"])

daily_demand.head()

,ProductID,Date,Demand
0,0128933d-9879-4346-aafc-a8045c36b087,2026-01-15,2
1,0128933d-9879-4346-aafc-a8045c36b087,2026-01-16,6
2,0128933d-9879-4346-aafc-a8045c36b087,2026-01-17,2
3,0128933d-9879-4346-aafc-a8045c36b087,2026-01-19,3
4,0128933d-9879-4346-aafc-a8045c36b087,2026-01-24,3


In [31]:
price_demand = data.groupby(
    ["ProductID", "EffectivePrice"]
).agg(
    TotalDemand=("Quantity", "sum"),
    AverageDiscount=("Discount", "mean")
).reset_index()

price_demand.head()

,ProductID,EffectivePrice,TotalDemand,AverageDiscount
0,0128933d-9879-4346-aafc-a8045c36b087,29113.0,1,4000.0
1,0128933d-9879-4346-aafc-a8045c36b087,29117.0,1,2500.0
2,0128933d-9879-4346-aafc-a8045c36b087,30027.0,2,4000.0
3,0128933d-9879-4346-aafc-a8045c36b087,30160.0,2,2500.0
4,0128933d-9879-4346-aafc-a8045c36b087,30184.0,2,3000.0


In [32]:
basket_data = data[
    ["InvoiceID", "ProductName"]
].drop_duplicates()

basket_data.head()

,InvoiceID,ProductName
0,INV-2026-1001,Merino Wool Minimalist Crewneck Knitwear
1,INV-2026-1001,Bio-Peptide Restorative Night Crème
2,INV-2026-1001,Velocity Pro Carbon Running Sneakers
3,INV-2026-1001,Ergonomic Breathable Mesh Desk Chair
5,INV-2026-1001,All-Terrain 4-Season Ultralight Tent


In [33]:
basket = pd.crosstab(
    basket_data["InvoiceID"],
    basket_data["ProductName"]
)

basket = basket.astype(bool)

basket.head()

ProductName,Acacia Wood Breakfast Tray,ActiveSense GPS Smart Band,Adventure Dome Camping Shelter,AeroFlex Performance Training Top,AeroFlex Seamless High-Rise Leggings,AeroStride Performance Running Shoes,AeroTrail Waterproof Hiking Backpack,AeroWatch AMOLED Fitness Edition,AirTone Compact Wireless Earbuds,All-Terrain 4-Season Ultralight Tent,...,Urban Classic Linen Overshirt,UrbanFlex Everyday Trainers,Velocity Lightweight Trail Sneakers,Velocity Pro Carbon Running Sneakers,Velvet Rose Evening Parfum,Viper Wireless 8K Hz Esports Mouse,VitaTrack Health Smartwatch,Vitamin C Brightening Moisturizer,WarmGlow Minimalist Wall Sconce,White Tea Botanical Fragrance
InvoiceID,,,,,,,,,,,,,,,,,,,,,
INV-2026-1001,False,False,False,False,False,False,False,False,False,True,...,False,False,False,True,False,False,False,False,False,False
INV-2026-1002,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
INV-2026-1003,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
INV-2026-1004,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
INV-2026-1005,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [34]:
feature_file = "C:/Users/RSCH/Desktop/DWD/Final project/Retail-Intelligence-Platform/datasets/amez_dataset_feature_engineered.xlsx"

data.to_excel(
    feature_file,
    sheet_name="Feature_Data",
    index=False
)

print("Feature engineering completed successfully.")

Feature engineering completed successfully.


In [35]:
daily_demand = daily_demand.sort_values(
    ["ProductID", "Date"]
)

daily_demand["Lag_1"] = (
    daily_demand.groupby("ProductID")["Demand"].shift(1)
)

daily_demand["Lag_7"] = (
    daily_demand.groupby("ProductID")["Demand"].shift(7)
)

daily_demand["RollingMean_7"] = (
    daily_demand.groupby("ProductID")["Demand"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

In [36]:
elasticity_data = data[
    [
        "ProductID",
        "InvoiceDate",
        "Quantity",
        "UnitPrice",
        "Discount",
        "EffectivePrice"
    ]
].copy()

In [37]:
elasticity_data = elasticity_data.sort_values(
    ["ProductID", "InvoiceDate"]
)

elasticity_data["PriceChange"] = (
    elasticity_data.groupby("ProductID")["EffectivePrice"]
    .pct_change()
)

elasticity_data["DemandChange"] = (
    elasticity_data.groupby("ProductID")["Quantity"]
    .pct_change()
)

In [38]:
data["StockStatus"] = np.where(
    data["StockQuantity"] <= 10,
    "Low Stock",
    "Available"
)

In [39]:
# Save complete feature-engineered transaction data
feature_file = "C:/Users/RSCH/Desktop/DWD/Final project/Retail-Intelligence-Platform/datasets/amez_dataset_feature_engineered.xlsx"

data.to_excel(
    feature_file,
    sheet_name="Feature_Data",
    index=False
)

In [40]:

customer_features.to_excel(
    "../datasets/customer_features.xlsx",
    index=False
)

product_features.to_excel(
    "../datasets/product_features.xlsx",
    index=False
)

daily_demand.to_excel(
    "../datasets/daily_demand_features.xlsx",
    index=False
)

elasticity_data.to_excel(
    "../datasets/elasticity_features.xlsx",
    index=False
)

basket_data.to_excel(
    "../datasets/basket_data.xlsx",
    index=False
)

customer_product.to_excel(
    "../datasets/customer_product_features.xlsx",
    index=False
)

print("Feature engineering completed successfully.")

Feature engineering completed successfully.
